# AlexNet (Paper-faithful) – Powerful Colab Quickstart

Clean modern PyTorch implementation of AlexNet with optional Local Response Normalization + **Grad-CAM**.

[GitHub repo](https://github.com/Stanley03-arch/alexnet-pytorch)

In [ ]:
# Install
!pip install -q torch torchvision matplotlib
!pip install -q git+https://github.com/Stanley03-arch/alexnet-pytorch.git

In [ ]:
from alexnet import AlexNet, summary, count_parameters, count_flops, GradCAM, overlay_cam_on_image
import torch
from PIL import Image
import matplotlib.pyplot as plt

# Modern variant (default)
model = AlexNet(num_classes=1000)
model.eval()

# Paper-faithful variant with Local Response Normalization
model_lrn = AlexNet(num_classes=1000, use_lrn=True)

print(f"Parameters: {count_parameters(model):,}")
print(f"FLOPs: {count_flops(model) / 1e9:.2f} GFLOPs")
summary(model)

In [ ]:
x = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    out = model(x)
print("Output shape:", out.shape)  # torch.Size([1, 1000])

In [ ]:
# Feature extraction for transfer learning
model.freeze_features()
features = model.extract_features(x)
print("Feature shape:", features.shape)  # (1, 9216)

## Grad-CAM Demo
Generate a visual explanation of what the model focuses on.

In [ ]:
# Create a simple synthetic image for demonstration
import numpy as np
from torchvision import transforms

# Random image as demo (replace with your own image for real results)
demo_img = Image.fromarray((np.random.rand(224, 224, 3) * 255).astype(np.uint8))

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
input_tensor = transform(demo_img).unsqueeze(0)

cam = GradCAM(model)
heatmap = cam(input_tensor)
overlay = overlay_cam_on_image(demo_img, heatmap)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(demo_img)
axes[0].set_title("Input")
axes[0].axis("off")
axes[1].imshow(overlay)
axes[1].set_title("Grad-CAM Overlay")
axes[1].axis("off")
plt.tight_layout()
plt.show()
print("Grad-CAM generated successfully!")